In [1]:
# Fair-Price Checker for Used Cars in India
# Algorithm: Random Forest Regression (Linear Regression as baseline)
# Dataset: Vehicle dataset from CarDekho (Kaggle) - Car details v3.csv

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# 2. Load the dataset
df = pd.read_csv("Car details v3.csv")
print("Shape of imported dataset:", df.shape)
df.head()

Shape of imported dataset: (8128, 13)


,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner,mileage,engine,max_power,torque,seats
0,Maruti Swift Dzire VDI,2014,450000,145500,Diesel,Individual,Manual,First Owner,23.4 kmpl,1248 CC,74 bhp,190Nm@ 2000rpm,5.0
1,Skoda Rapid 1.5 TDI Ambition,2014,370000,120000,Diesel,Individual,Manual,Second Owner,21.14 kmpl,1498 CC,103.52 bhp,250Nm@ 1500-2500rpm,5.0
2,Honda City 2017-2020 EXi,2006,158000,140000,Petrol,Individual,Manual,Third Owner,17.7 kmpl,1497 CC,78 bhp,"12.7@ 2,700(kgm@ rpm)",5.0
3,Hyundai i20 Sportz Diesel,2010,225000,127000,Diesel,Individual,Manual,First Owner,23.0 kmpl,1396 CC,90 bhp,22.4 kgm at 1750-2750rpm,5.0
4,Maruti Swift VXI BSIII,2007,130000,120000,Petrol,Individual,Manual,First Owner,16.1 kmpl,1298 CC,88.2 bhp,"11.5@ 4,500(kgm@ rpm)",5.0


In [3]:
# 3. Data preprocessing
CURRENT_YEAR = 2026
df = df.drop_duplicates()

# Clean string numeric columns
for col in ["mileage", "engine", "max_power"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col].astype(str).str.extract(r"([\d.]+)")[0], errors="coerce")

df["age"] = CURRENT_YEAR - df["year"]
df["brand"] = df["name"].str.split().str[0]
df = df.drop(columns=[c for c in ["name", "year", "torque"] if c in df.columns])
df = df.dropna()
print("Rows after cleaning:", len(df))

Rows after cleaning: 6717


In [4]:
# 5. Encode categorical features and split
y = df["selling_price"]
X = pd.get_dummies(df.drop(columns="selling_price"))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("Train size:", X_train.shape, "Test size:", X_test.shape)

Train size: (5373, 51) Test size: (1344, 51)


In [5]:
# 6. Train the Random Forest Model
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
print("Model training complete!")

Model training complete!


In [6]:
import os

file_name = "Car details v3.csv"
if os.path.exists(file_name):
    print(f"Success! '{file_name}' is uploaded and ready in the current workspace.")
else:
    print(f"No, '{file_name}' is NOT uploaded yet.")
    print("Please click the Folder icon on the left sidebar of Google Colab, and upload or drag-and-drop the CSV file there.")

Success! 'Car details v3.csv' is uploaded and ready in the current workspace.


### Improved Fair-Price Estimator
Below is the updated `fair_price` function. It safely aligns unseen brand categories (and other categorical values) to the training feature space `X.columns` using `.reindex()`, and ensures that tree predictions are computed correctly.

In [7]:
import pandas as pd
import numpy as np
import warnings

def fair_price(car):
    # Convert the single input dictionary into a DataFrame
    row = pd.get_dummies(pd.DataFrame([car]))

    # Align columns with the training set features (X.columns)
    # Unseen categories will be dropped, and missing categories will be filled with 0
    if 'X' in globals():
        row = row.reindex(columns=X.columns, fill_value=0)

        # Check if the brand provided exists in our training columns
        brand_query = f"brand_{car.get('brand', '')}"
        if brand_query not in X.columns:
            print(f"Warning: The brand '{car.get('brand')}' was not in the training dataset. Predicting based on generic car attributes.")
    else:
        print("Warning: Training features 'X' not found in memory. Please run the notebook cells above to build and train the model first.")
        return

    if 'rf' in globals():
        # Obtain tree predictions securely using the matched DataFrame format
        # We use warnings.catch_warnings to suppress the scikit-learn feature name warnings cleanly
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", UserWarning)
            tree_preds = np.array([t.predict(row.values)[0] for t in rf.estimators_])

        low, mid, high = np.percentile(tree_preds, 10), tree_preds.mean(), np.percentile(tree_preds, 90)
        print(f"Estimated fair price : Rs {mid:,.0f}")
        print(f"Likely range         : Rs {low:,.0f}  to  Rs {high:,.0f}")
    else:
        print("Warning: Trained Random Forest model 'rf' not found in memory. Please run the model training cells above first.")

# Test with an unseen brand (e.g., 'Tesla') to demonstrate robustness
fair_price({
    "brand": "Tesla", "age": 3, "km_driven": 15000, "fuel": "Electric",
    "seller_type": "Individual", "transmission": "Automatic", "owner": "First Owner",
    "mileage": 15.0, "engine": 0, "max_power": 250, "seats": 5,
})

Estimated fair price : Rs 5,691,060
Likely range         : Rs 5,150,000  to  Rs 7,200,000


### Interactive Price Estimator
Run the cell below to answer a few quick questions about your car, and get an instant fair price estimation!

In [8]:
def interactive_estimator():
    print("--- Welcome to the Used Car Fair Price Estimator ---\n")

    # Helper to capture and clean inputs
    def get_input(prompt, default_val, is_numeric=False):
        val = input(f"{prompt} [Default: {default_val}]: ").strip()
        if not val:
            return default_val
        if is_numeric:
            try:
                return float(val) if '.' in val else int(val)
            except ValueError:
                print(f"Invalid input. Using default value: {default_val}")
                return default_val
        return val

    # Prompt user for attributes with fallback defaults
    car_name = get_input("1. Enter the specific model name (e.g., Swift Dzire VDI)", "Swift Dzire VDI")
    brand = get_input("2. Enter the car brand (e.g., Maruti, Hyundai, Tesla)", "Maruti")
    age = get_input("3. Enter the car age in years (e.g., 5)", 5, is_numeric=True)
    km_driven = get_input("4. Enter kilometers driven (e.g., 40000)", 40000, is_numeric=True)
    fuel = get_input("5. Enter fuel type (Diesel / Petrol / CNG / LPG / Electric)", "Petrol")
    seller_type = get_input("6. Enter seller type (Individual / Dealer / Trustmark Dealer)", "Individual")
    transmission = get_input("7. Enter transmission (Manual / Automatic)", "Manual")
    owner = get_input("8. Enter owner type (First Owner / Second Owner / Third Owner)", "First Owner")
    mileage = get_input("9. Enter mileage in kmpl (e.g., 18.2)", 18.2, is_numeric=True)
    engine = get_input("10. Enter engine capacity in CC (e.g., 1197)", 1197, is_numeric=True)
    max_power = get_input("11. Enter max power in bhp (e.g., 82)", 82, is_numeric=True)
    seats = get_input("12. Enter number of seats (e.g., 5)", 5, is_numeric=True)

    # Extended fields regarding vehicle condition
    body_condition = get_input("13. Describe the outer body condition (e.g., Excellent, minor scratches, dent on left door)", "Excellent")
    maintenance_issues = get_input("14. Mention any current maintenance/mechanical problems (or 'None')", "None")

    user_car = {
        "brand": brand,
        "age": age,
        "km_driven": km_driven,
        "fuel": fuel,
        "seller_type": seller_type,
        "transmission": transmission,
        "owner": owner,
        "mileage": mileage,
        "engine": engine,
        "max_power": max_power,
        "seats": seats
    }

    print("\n=========================================")
    print("           VEHICLE REPORT CARD           ")
    print("=========================================")
    print(f"Car Model     : {brand} {car_name}")
    print(f"Outer Body    : {body_condition}")
    print(f"Issues/Faults : {maintenance_issues}")
    print("-----------------------------------------")
    print("Running ML Estimation Pipeline...")
    fair_price(user_car)
    print("=========================================")

# Run the updated questionnaire
interactive_estimator()

--- Welcome to the Used Car Fair Price Estimator ---

1. Enter the specific model name (e.g., Swift Dzire VDI) [Default: Swift Dzire VDI]: City ZX
2. Enter the car brand (e.g., Maruti, Hyundai, Tesla) [Default: Maruti]: Honda
3. Enter the car age in years (e.g., 5) [Default: 5]: 8
4. Enter kilometers driven (e.g., 40000) [Default: 40000]: 80000
5. Enter fuel type (Diesel / Petrol / CNG / LPG / Electric) [Default: Petrol]: Petrol
6. Enter seller type (Individual / Dealer / Trustmark Dealer) [Default: Individual]: Individual
7. Enter transmission (Manual / Automatic) [Default: Manual]: Automatic
8. Enter owner type (First Owner / Second Owner / Third Owner) [Default: First Owner]: First Owner
9. Enter mileage in kmpl (e.g., 18.2) [Default: 18.2]: 12
10. Enter engine capacity in CC (e.g., 1197) [Default: 1197]: 1197
11. Enter max power in bhp (e.g., 82) [Default: 82]: 117
12. Enter number of seats (e.g., 5) [Default: 5]: 5
13. Describe the outer body condition (e.g., Excellent, minor scra

### 🛠️ Car Value-Adding Modifications & Value Upgrades Guide

While the machine learning model estimates the baseline market value based on standard factory specifications, real-world buyer premiums depend highly on aftermarket modifications. Here is a guide on common modifications and their typical impact on resale value in the Indian used car market:

1. **Premium Alloy Wheels & Tyres** (Value Increase: **Rs 15,000 - Rs 35,000**)
   * Upgrading to clean, high-profile alloys or brand-new premium tyres increases aesthetic appeal and road handling.

2. **Upgraded Infotainment & Audio System** (Value Increase: **Rs 10,000 - Rs 25,000**)
   * Adding a responsive Android/Apple CarPlay touchscreen head unit, reverse parking camera, and high-quality speakers.

3. **Leather or Premium Upholstery** (Value Increase: **Rs 5,000 - Rs 15,000**)
   * Clean, customized leatherette or premium fabric seat covers instantly elevate cabin freshness and mask age.

4. **Paint Protection Film (PPF) or Ceramic Coating** (Value Increase: **Rs 8,000 - Rs 20,000**)
   * A gloss-retaining ceramic treatment or protective film shows the vehicle's paint has been meticulously cared for, allowing a premium ask.

5. **CNG Kit Installation (RTO Approved)** (Value Increase: **Rs 20,000 - Rs 40,000** for Petrol cars)
   * Highly desirable for commuter cars in metropolitan areas due to massive savings on fuel costs.

In [9]:
def advanced_interactive_estimator():
    import pandas as pd
    import numpy as np
    import warnings

    print("--- Advanced Used Car Fair Price & On-Road Cost Comparison ---\n")

    def get_input(prompt, default_val, is_numeric=False):
        val = input(f"{prompt} [Default: {default_val}]: ").strip()
        if not val:
            return default_val
        if is_numeric:
            try:
                return float(val) if '.' in val else int(val)
            except ValueError:
                print(f"Invalid input. Using default value: {default_val}")
                return default_val
        return val

    # Core ML specs
    car_name = get_input("1. Specific model name (e.g., City ZX, Innova, Swift)", "City ZX")
    brand = get_input("2. Brand (e.g., Toyota, Honda, Maruti, Hyundai)", "Honda")
    age = get_input("3. Car age (in years)", 5, is_numeric=True)
    km_driven = get_input("4. Odometer reading (km)", 30000, is_numeric=True)
    fuel = get_input("5. Fuel type (Diesel / Petrol / CNG / LPG / Electric)", "Petrol")
    seller_type = get_input("6. Seller Type (Individual / Dealer / Trustmark Dealer)", "Individual")
    transmission = get_input("7. Transmission (Manual / Automatic)", "Automatic")
    owner = get_input("8. Owner status (First Owner / Second Owner / Third Owner)", "First Owner")
    mileage = get_input("9. Fuel Economy / Mileage in kmpl (e.g. 18.0)", 11.0, is_numeric=True)
    engine = get_input("10. Engine Capacity in CC (e.g., 1497, 2755)", 1497, is_numeric=True)
    max_power = get_input("11. Max Power in bhp (e.g., 117, 172)", 117, is_numeric=True)
    seats = get_input("12. Number of seats", 5, is_numeric=True)

    # Small, easy questions to compute a better body condition assessment
    print("\n--- Vehicle Body & Paint Assessment ---")
    scratches = get_input("13. Does the car have any visible scratches or dents? (Yes/No)", "No")
    paint_fade = get_input("14. Is there any noticeable paint fading or panels repainted? (Yes/No)", "No")
    accidental = get_input("15. Has the car ever had any structural or accidental damage? (Yes/No)", "No")

    # Mechanical & Documentation Checks
    maintenance_issues = get_input("16. Any current mechanical/engine problems? (e.g. None)", "None")
    service_history = get_input("17. Service History (Authorized Dealer / Local Mechanic)", "Authorized Dealer Records")
    insurance_valid = get_input("18. Insurance validity (Comprehensive / Third Party / Expired)", "Comprehensive")
    tyre_condition = get_input("19. Tyre condition (New / 50% worn / Replacement needed)", "New")

    # Scoring body condition dynamically
    body_penalty = 0.0
    body_details = []
    if scratches.lower() in ["yes", "y"]:
        body_penalty += 0.05
        body_details.append("Minor Scratches/Dents")
    if paint_fade.lower() in ["yes", "y"]:
        body_penalty += 0.07
        body_details.append("Paint imperfections/Repaint")
    if accidental.lower() in ["yes", "y"]:
        body_penalty += 0.25
        body_details.append("Accidental Damage/History")

    body_score_label = "Excellent" if body_penalty == 0.0 else ("Good" if body_penalty <= 0.10 else "Fair")
    if body_penalty >= 0.25:
        body_score_label = "Accident Recovered / Poor"

    # Modifications input
    print("\nHave you installed any premium modifications? (Enter 'Yes' or 'No')")
    has_mods = get_input("Has Modifications?", "No")
    mod_bonus = 0
    mod_details = []

    if has_mods.lower() in ["yes", "y"]:
        print("\nSelect upgrades to apply value adjustment premium:")
        alloys = get_input("- Upgraded Alloy Wheels & Tyres? (Yes/No)", "No")
        infotainment = get_input("- Custom Touchscreen/Audio Upgrades? (Yes/No)", "No")
        upholstery = get_input("- Premium Leather Seat Covers? (Yes/No)", "No")
        cng_kit = get_input("- Government Approved Aftermarket CNG Kit? (Yes/No)", "No")

        if alloys.lower() in ["yes", "y"]:
            mod_bonus += 25000
            mod_details.append("Alloy Wheels & Premium Tyres (+Rs 25,000)")
        if infotainment.lower() in ["yes", "y"]:
            mod_bonus += 15000
            mod_details.append("Premium Touchscreen & Audio (+Rs 15,000)")
        if upholstery.lower() in ["yes", "y"]:
            mod_bonus += 10000
            mod_details.append("Premium Leather Upholstery (+Rs 10,000)")
        if cng_kit.lower() in ["yes", "y"] and fuel.lower() != "cng":
            mod_bonus += 30000
            mod_details.append("RTO Approved CNG Kit (+Rs 30,000)")

    user_car = {
        "brand": brand, "age": age, "km_driven": km_driven, "fuel": fuel,
        "seller_type": seller_type, "transmission": transmission, "owner": owner,
        "mileage": mileage, "engine": engine, "max_power": max_power, "seats": seats
    }

    # Segment-based dynamic Indian Ex-Showroom pricing estimation
    base_showroom = 600000
    brand_lower = brand.lower()
    name_lower = car_name.lower()

    if brand_lower in ["bmw", "mercedes", "audi", "jaguar", "land", "volvo"]:
        base_showroom = 4500000
    elif "innova" in name_lower or "fortuner" in name_lower:
        base_showroom = 2400000 if "fortuner" in name_lower else 1900000
    elif engine >= 2000:
        base_showroom = 2200000 if brand_lower in ["toyota", "jeep"] else 1600000
    elif engine >= 1400:
        base_showroom = 1350000 if transmission.lower() == "automatic" else 1150000
    elif engine >= 1100 or "venue" in name_lower or "brezza" in name_lower:
        base_showroom = 950000

    # Add RTO (Road Tax ~12%), Insurance (~4%), and TCS / Depot charges (~1.5%) to find On-Road Price
    rto_tax = base_showroom * 0.12
    insurance_cost = base_showroom * 0.04
    tcs_misc = base_showroom * 0.015
    estimated_on_road_new = base_showroom + rto_tax + insurance_cost + tcs_misc

    print("\n=======================================================")
    print("         REAL-WORLD ON-ROAD PRICE COMPARISON           ")
    print("=======================================================")
    print(f"Car Model          : {brand} {car_name} ({fuel}, {transmission})")
    print(f"Vehicle Age / Run  : {age} Years / {km_driven:,} km")
    print(f"Est. Original Brand-New Ex-Showroom : Rs {base_showroom:,.0f}")
    print(f"Est. Original Brand-New ON-ROAD Cost: Rs {estimated_on_road_new:,.0f}")
    print("-------------------------------------------------------")
    print("Calculating Present Day ML Valuation...")

    # Retrieve baseline price predictions
    row = pd.get_dummies(pd.DataFrame([user_car]))
    if 'X' in globals() and 'rf' in globals():
        row = row.reindex(columns=X.columns, fill_value=0)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", UserWarning)
            tree_preds = np.array([t.predict(row.values)[0] for t in rf.estimators_])

        # Tighter 25th - 75th percentile range to prevent way-off target brackets
        low_base, mid_base, high_base = np.percentile(tree_preds, 25), tree_preds.mean(), np.percentile(tree_preds, 75)

        # Apply body condition penalty
        mid_base = mid_base * (1.0 - body_penalty)
        low_base = low_base * (1.0 - body_penalty)
        high_base = high_base * (1.0 - body_penalty)

        # Add modifications premiums safely
        low_final = min(low_base + mod_bonus, estimated_on_road_new * 0.95)
        mid_final = min(mid_base + mod_bonus, estimated_on_road_new * 0.90)
        high_final = min(high_base + mod_bonus, estimated_on_road_new * 0.98)

        total_depreciation = ((estimated_on_road_new - mid_final) / estimated_on_road_new) * 100

        print(f"Assessed Body Condition    : {body_score_label} (Total Penalty: -{body_penalty*100:.0f}%)")
        if body_details:
            print(f"Detected Detractions       : {', '.join(body_details)}")
        print(f"Baseline Used Valuation     : Rs {mid_base:,.0f}")
        print(f"Modifications Bonus Added   : Rs {mod_bonus:,.0f}")
        print(f"Current Fair Market Resale  : Rs {mid_final:,.0f}")
        print(f"Total Depreciation Loss     : {total_depreciation:.1f}% off original on-road cost")
        print(f"Target Resale Bracket (25-75): Rs {low_final:,.0f}  to  Rs {high_final:,.0f}")
    else:
        print("\nError: ML Models not loaded in active memory. Please run cell 'b656651b' first to automatically reload variables and train the Random Forest!")
    print("=======================================================")

# Run advanced system
advanced_interactive_estimator()

--- Advanced Used Car Fair Price & On-Road Cost Comparison ---

1. Specific model name (e.g., City ZX, Innova, Swift) [Default: City ZX]: City ZX
2. Brand (e.g., Toyota, Honda, Maruti, Hyundai) [Default: Honda]: Honda
3. Car age (in years) [Default: 5]: 8
4. Odometer reading (km) [Default: 30000]: 80000
5. Fuel type (Diesel / Petrol / CNG / LPG / Electric) [Default: Petrol]: Petrol
6. Seller Type (Individual / Dealer / Trustmark Dealer) [Default: Individual]: Individual
7. Transmission (Manual / Automatic) [Default: Automatic]: Automatic
8. Owner status (First Owner / Second Owner / Third Owner) [Default: First Owner]: First owner
9. Fuel Economy / Mileage in kmpl (e.g. 18.0) [Default: 11.0]: 12
10. Engine Capacity in CC (e.g., 1497, 2755) [Default: 1497]: 1197
11. Max Power in bhp (e.g., 117, 172) [Default: 117]: 117
12. Number of seats [Default: 5]: 5

--- Vehicle Body & Paint Assessment ---
13. Does the car have any visible scratches or dents? (Yes/No) [Default: No]: Yes
14. Is ther

In [10]:
import numpy as np
import pandas as pd
import warnings
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

# Quick auto-setup to restore notebook state in memory
CURRENT_YEAR = 2026
try:
    df = pd.read_csv("Car details v3.csv")
    df = df.drop_duplicates()

    for col in ["mileage", "engine", "max_power"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col].astype(str).str.extract(r"([\d.]+)")[0], errors="coerce")

    df["age"] = CURRENT_YEAR - df["year"]
    df["brand"] = df["name"].str.split().str[0]
    df = df.drop(columns=[c for c in ["name", "year", "torque"] if c in df.columns])
    df = df.dropna()

    y = df["selling_price"]
    X = pd.get_dummies(df.drop(columns="selling_price"))

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    print("Dependencies restored: 'pd', 'X', and trained model 'rf' are ready in memory!")
except Exception as e:
    print(f"Could not auto-train. Please make sure 'Car details v3.csv' is uploaded. Error: {e}")

Dependencies restored: 'pd', 'X', and trained model 'rf' are ready in memory!


### 🚀 Model Optimization & Error Reduction
To reduce prediction error, we perform advanced feature engineering (applying a logarithmic transform to the skewed target variable `selling_price`) and optimize the Random Forest Regressor hyperparameters.

In [11]:
import numpy as np
import pandas as pd
import warnings
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# Prepare data with log-transformed target to handle large outliers and reduce error
y_log = np.log1p(y)
X_train, X_test, y_train_log, y_test_log = train_test_split(
    X, y_log, test_size=0.2, random_state=42
)

# Train optimized Random Forest Regressor
rf_optimized = RandomForestRegressor(
    n_estimators=150,
    max_depth=20,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)
rf_optimized.fit(X_train, y_train_log)

# Evaluate optimized model
y_pred_log = rf_optimized.predict(X_test)
y_pred = np.expm1(y_pred_log)

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Optimized Model Mean Absolute Error (MAE): Rs {mae:,.2f}")
print(f"Optimized Model R-squared (R2) Score: {r2:.4f}")

Optimized Model Mean Absolute Error (MAE): Rs 72,543.11
Optimized Model R-squared (R2) Score: 0.9301


### 🚀 Performance Optimization: Feature Engineering & LightGBM
To push the accuracy higher, we can perform target-encoding on the `brand` feature and train a **LightGBM** model. Gradient boosted trees usually capture complex feature relationships much better than basic Random Forests.

In [12]:
# Calculate brand mean prices from the training partition to avoid target leakage
df_train_orig, df_test_orig = train_test_split(df, test_size=0.2, random_state=42)
brand_means = df_train_orig.groupby('brand')['selling_price'].mean().to_dict()

# Map brand means as a new numerical feature representing brand segment tier
df['brand_tier_price'] = df['brand'].map(brand_means).fillna(df_train_orig['selling_price'].mean())

# Power-to-Engine ratio (proxy for performance metrics)
df['power_to_engine'] = df['max_power'] / (df['engine'] + 1)

# Prepare features
y_opt = df['selling_price']
X_opt = pd.get_dummies(df.drop(columns='selling_price'))

y_opt_log = np.log1p(y_opt)
X_train_opt, X_test_opt, y_train_opt_log, y_test_opt_log = train_test_split(
    X_opt, y_opt_log, test_size=0.2, random_state=42
)

print("New features engineered successfully! Ready to fit LightGBM.")

New features engineered successfully! Ready to fit LightGBM.


In [13]:
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, r2_score

# Configure LightGBM Regressor with regularized parameters to prevent overfitting
lgb_model = lgb.LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbose=-1
)

lgb_model.fit(X_train_opt, y_train_opt_log)

# Evaluate the new model
y_pred_lgb_log = lgb_model.predict(X_test_opt)
y_pred_lgb = np.expm1(y_pred_lgb_log)
y_test_opt = np.expm1(y_test_opt_log)

lgb_mae = mean_absolute_error(y_test_opt, y_pred_lgb)
lgb_r2 = r2_score(y_test_opt, y_pred_lgb)

print(f"Optimized LightGBM Mean Absolute Error (MAE): Rs {lgb_mae:,.2f}")
print(f"Optimized LightGBM R-squared (R2) Score: {lgb_r2:.4f}")

Optimized LightGBM Mean Absolute Error (MAE): Rs 71,239.62
Optimized LightGBM R-squared (R2) Score: 0.9322


### 🤝 Deal Evaluation & Interactive Estimator
Now we update the interactive estimator to accept an offered price and evaluate whether it represents a good deal compared to the fair market value predicted by our model.

In [ ]:
def interactive_estimator_with_deal_check():
    print("--- Welcome to the Used Car Fair Price Estimator & Deal Evaluator ---\n")

    def get_input(prompt, default_val, is_numeric=False):
        val = input(f"{prompt} [Default: {default_val}]: ").strip()
        if not val:
            return default_val
        if is_numeric:
            try:
                return float(val) if '.' in val else int(val)
            except ValueError:
                print(f"Invalid input. Using default value: {default_val}")
                return default_val
        return val

    # Collect core user specs
    brand = get_input("1. Enter the car brand (e.g., Maruti, Honda, Toyota)", "Honda")
    age = get_input("2. Enter the car age in years", 5, is_numeric=True)
    km_driven = get_input("3. Enter kilometers driven", 40000, is_numeric=True)
    fuel = get_input("4. Enter fuel type (Diesel / Petrol / CNG / LPG / Electric)", "Petrol")
    seller_type = get_input("5. Enter seller type (Individual / Dealer / Trustmark Dealer)", "Individual")
    transmission = get_input("6. Enter transmission (Manual / Automatic)", "Manual")
    owner = get_input("7. Enter owner type (First Owner / Second Owner / Third Owner)", "First Owner")
    mileage = get_input("8. Enter mileage in kmpl", 18.2, is_numeric=True)
    engine = get_input("9. Enter engine capacity in CC", 1197, is_numeric=True)
    max_power = get_input("10. Enter max power in bhp", 82, is_numeric=True)
    seats = get_input("11. Enter number of seats", 5, is_numeric=True)

    # New input: Offered Price for assessment
    offered_price = get_input("12. What is the seller's offered price in Rs? (e.g., 450000)", 450000, is_numeric=True)

    user_car = {
        "brand": brand,
        "age": age,
        "km_driven": km_driven,
        "fuel": fuel,
        "seller_type": seller_type,
        "transmission": transmission,
        "owner": owner,
        "mileage": mileage,
        "engine": engine,
        "max_power": max_power,
        "seats": seats
    }

    # Prepare row for prediction
    row = pd.get_dummies(pd.DataFrame([user_car]))
    row = row.reindex(columns=X.columns, fill_value=0)

    # Predict using optimized model estimators
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)
        tree_preds_log = np.array([t.predict(row.values)[0] for t in rf_optimized.estimators_])
        tree_preds = np.expm1(tree_preds_log)

    mid_val = float(tree_preds.mean())
    low_val = float(np.percentile(tree_preds, 10))
    high_val = float(np.percentile(tree_preds, 90))

    print("\n=========================================")
    print("           VALUATION & DEAL REPORT        ")
    print("=========================================")
    print(f"Estimated Fair Price : Rs {mid_val:,.0f}")
    print(f"Likely Range         : Rs {low_val:,.0f} to Rs {high_val:,.0f}")
    print(f"Your Offered Price   : Rs {offered_price:,.0f}")
    print("-----------------------------------------")

    # Determine deal health
    price_diff_pct = ((offered_price - mid_val) / mid_val) * 100

    if offered_price < low_val:
        print("🔥 Verdict: EXCELLENT DEAL! The offered price is well below the market range.")
    elif offered_price <= mid_val:
        print("✅ Verdict: GOOD DEAL! The price is below or match the average market value.")
    elif offered_price <= high_val:
        print("🤝 Verdict: FAIR PRICE. The price is slightly above average, but acceptable.")
    else:
        print(f"❌ Verdict: OVERPRICED. The seller's asking price is {price_diff_pct:.1f}% above our fair valuation.")
    print("=========================================")

# Run the updated evaluator
interactive_estimator_with_deal_check()

--- Welcome to the Used Car Fair Price Estimator & Deal Evaluator ---



### 🎛️ Safe & Interactive Form-Based Estimator
Rather than using blocking `input()` prompts (which cause runtime issues), fill out the parameters on the right-hand form panel and run the cell to instantly calculate your car's valuation and deal status.

In [ ]:
#@title 🚗 Used Car Deal Evaluator Form
import pandas as pd
import numpy as np
import warnings

#@markdown ### 1. Core Specifications
brand = "Honda" #@param {type:"string"}
car_name = "City ZX" #@param {type:"string"}
age = 8 #@param {type:"slider", min:0, max:30, step:1}
km_driven = 80000 #@param {type:"integer"}
fuel = "Petrol" #@param ["Diesel", "Petrol", "CNG", "LPG", "Electric"]
seller_type = "Individual" #@param ["Individual", "Dealer", "Trustmark Dealer"]
transmission = "Automatic" #@param ["Manual", "Automatic"]
owner = "First Owner" #@param ["First Owner", "Second Owner", "Third Owner", "Fourth & Above Owner"]

#@markdown ### 2. Technical Metrics
mileage = 12.0 #@param {type:"number"}
engine = 1197 #@param {type:"integer"}
max_power = 117 #@param {type:"integer"}
seats = 5 #@param {type:"integer"}

#@markdown ### 3. Deal Details
offered_price = 450000 #@param {type:"integer"}

def form_deal_evaluator():
    user_car = {
        "brand": brand,
        "age": age,
        "km_driven": km_driven,
        "fuel": fuel,
        "seller_type": seller_type,
        "transmission": transmission,
        "owner": owner,
        "mileage": mileage,
        "engine": engine,
        "max_power": max_power,
        "seats": seats
    }

    # Prepare row for prediction
    row = pd.get_dummies(pd.DataFrame([user_car]))
    row = row.reindex(columns=X.columns, fill_value=0)

    # Predict using optimized model estimators
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)
        tree_preds_log = np.array([t.predict(row.values)[0] for t in rf_optimized.estimators_])
        tree_preds = np.expm1(tree_preds_log)

    mid_val = float(tree_preds.mean())
    low_val = float(np.percentile(tree_preds, 10))
    high_val = float(np.percentile(tree_preds, 90))

    print("=========================================")
    print("           VALUATION & DEAL REPORT        ")
    print("=========================================")
    print(f"Car Model            : {brand} {car_name}")
    print(f"Estimated Fair Price : Rs {mid_val:,.0f}")
    print(f"Likely Range         : Rs {low_val:,.0f} to Rs {high_val:,.0f}")
    print(f"Your Offered Price   : Rs {offered_price:,.0f}")
    print("-----------------------------------------")

    # Determine deal health
    price_diff_pct = ((offered_price - mid_val) / mid_val) * 100

    if offered_price < low_val:
        print("🔥 Verdict: EXCELLENT DEAL! The offered price is well below the market range.")
    elif offered_price <= mid_val:
        print("✅ Verdict: GOOD DEAL! The price is below or match the average market value.")
    elif offered_price <= high_val:
        print("🤝 Verdict: FAIR PRICE. The price is slightly above average, but acceptable.")
    else:
        print(f"❌ Verdict: OVERPRICED. The seller's asking price is {price_diff_pct:.1f}% above our fair valuation.")
    print("=========================================")

form_deal_evaluator()

### 🧙‍♂️ Live Multi-Step Car Valuation Wizard (with Back/Next Navigation)
Run the cell below to load an interactive step-by-step dashboard. You can navigate freely between steps, adjust metrics, and get instant, highly accurate valuations!

In [ ]:
import IPython
from google.colab import output
import pandas as pd
import numpy as np
import warnings

# Advanced prediction function using engineered features and LightGBM model for highest accuracy
def predict_wizard_valuation(brand, age, km_driven, fuel, seller_type, transmission, owner, mileage, engine, max_power, seats, offered_price):
    user_car = {
        "brand": brand,
        "age": float(age),
        "km_driven": float(km_driven),
        "fuel": fuel,
        "seller_type": seller_type,
        "transmission": transmission,
        "owner": owner,
        "mileage": float(mileage),
        "engine": float(engine),
        "max_power": float(max_power),
        "seats": float(seats)
    }

    # Generate engineered features online for incoming user data
    # 1. Target encoded brand tier
    brand_mapped_price = brand_means.get(brand, df_train_orig['selling_price'].mean())
    user_car['brand_tier_price'] = brand_mapped_price

    # 2. Power to Engine ratio
    user_car['power_to_engine'] = float(max_power) / (float(engine) + 1)

    # Prepare features and map dummy columns aligned with the LightGBM training schema
    row = pd.get_dummies(pd.DataFrame([user_car]))
    row = row.reindex(columns=X_opt.columns, fill_value=0)

    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)
        if 'lgb_model' in globals() and lgb_model is not None:
            pred_log = lgb_model.predict(row)
            mid_val = float(np.expm1(pred_log)[0])
        else:
            # Fallback to RF if LGB is not initialized
            row_rf = row.reindex(columns=X.columns, fill_value=0)
            pred_log = rf_optimized.predict(row_rf.values)
            mid_val = float(np.expm1(pred_log)[0])

    # Set realistic range bounds
    low_val = mid_val * 0.85
    high_val = mid_val * 1.15
    price_diff_pct = ((float(offered_price) - mid_val) / mid_val) * 100

    # Pick structural verdict text
    if float(offered_price) < low_val:
        verdict = f"🔥 EXCELLENT DEAL! The offered price is well below the market range."
    elif float(offered_price) <= mid_val:
        verdict = f"✅ GOOD DEAL! The price is below or match the average market value."
    elif float(offered_price) <= high_val:
        verdict = f"🤝 FAIR PRICE. The price is slightly above average, but acceptable."
    else:
        verdict = f"❌ OVERPRICED. The seller's asking price is {price_diff_pct:.1f}% above our fair valuation."

    return {
        "mid": f"Rs {mid_val:,.0f}",
        "range": f"Rs {low_val:,.0f} to Rs {high_val:,.0f}",
        "verdict": verdict
    }

# Register callback function to bridge Python prediction and JavaScript interactive UI
output.register_callback('notebook.predict_wizard_valuation', predict_wizard_valuation)

# Inject UI directly using HTML & JavaScript wizard
html_content = """
<!DOCTYPE html>
<html>
<head>
<style>
  .wizard-card {
    font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
    background: #ffffff;
    border-radius: 12px;
    box-shadow: 0 4px 20px rgba(0,0,0,0.1);
    padding: 24px;
    max-width: 600px;
    margin: 10px auto;
    border: 1px solid #e0e0e0;
  }
  .wizard-header {
    border-bottom: 2px solid #f0f0f0;
    padding-bottom: 12px;
    margin-bottom: 20px;
  }
  .wizard-header h3 { margin: 0; color: #1a73e8; }
  .step-container { display: none; }
  .step-container.active { display: block; }
  .form-group {
    margin-bottom: 16px;
  }
  .form-group label {
    display: block;
    font-weight: 600;
    margin-bottom: 6px;
    color: #333;
  }
  .form-group input, .form-group select {
    width: 100%;
    padding: 10px;
    border-radius: 6px;
    border: 1px solid #ccc;
    box-sizing: border-box;
    font-size: 14px;
  }
  .btn-container {
    display: flex;
    justify-content: space-between;
    margin-top: 24px;
  }
  button {
    padding: 10px 20px;
    border: none;
    border-radius: 6px;
    cursor: pointer;
    font-weight: 600;
    font-size: 14px;
  }
  .btn-next { background: #1a73e8; color: white; }
  .btn-back { background: #f1f3f4; color: #3c4043; }
  .btn-submit { background: #34a853; color: white; }
  .progress-bar {
    height: 6px;
    background: #f1f3f4;
    border-radius: 3px;
    margin-bottom: 20px;
    overflow: hidden;
  }
  .progress-fill {
    height: 100%;
    background: #1a73e8;
    width: 25%;
    transition: width 0.3s ease;
  }
  #resultPanel {
    background: #f8f9fa;
    padding: 16px;
    border-radius: 8px;
    border-left: 5px solid #34a853;
    margin-top: 20px;
    display: none;
  }
</style>
</head>
<body>

<div class="wizard-card">
  <div class="wizard-header">
    <h3>🚗 Smart Valuation & Deal Evaluator Wizard</h3>
    <p style="color:#666; font-size:13px;">Fill out specs step-by-step. You can navigate back at any time to adjust your selections.</p>
  </div>

  <div class="progress-bar">
    <div class="progress-fill" id="progressFill"></div>
  </div>

  <!-- Step 1 -->
  <div class="step-container active" id="step1">
    <h4>Step 1: Core Identification</h4>
    <div class=

In [ ]:
#@title 🚗 High-Accuracy Used Car Deal Evaluator Dashboard
import pandas as pd
import numpy as np
import warnings
from sklearn.model_selection import train_test_split
import lightgbm as lgb
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# --- Step 1: Model Setup & In-Memory Calibration (Runs once) ---
CURRENT_YEAR = 2026
try:
    # Load and prep dataset
    wiz_df = pd.read_csv("Car details v3.csv").drop_duplicates()

    for col in ["mileage", "engine", "max_power"]:
        if col in wiz_df.columns:
            wiz_df[col] = pd.to_numeric(wiz_df[col].astype(str).str.extract(r"([\d.]+)")[0], errors="coerce")

    wiz_df["age"] = CURRENT_YEAR - wiz_df["year"]
    wiz_df["brand"] = wiz_df["name"].str.split().str[0]
    wiz_df = wiz_df.drop(columns=[c for c in ["name", "year", "torque"] if c in wiz_df.columns]).dropna()

    # Feature Engineering and Target Encoding in-place to keep cell independent
    df_train_orig, df_test_orig = train_test_split(wiz_df, test_size=0.2, random_state=42)
    brand_means = df_train_orig.groupby('brand')['selling_price'].mean().to_dict()

    wiz_df['brand_tier_price'] = wiz_df['brand'].map(brand_means).fillna(df_train_orig['selling_price'].mean())
    wiz_df['power_to_engine'] = wiz_df['max_power'] / (wiz_df['engine'] + 1)

    y_opt = wiz_df['selling_price']
    X_opt = pd.get_dummies(wiz_df.drop(columns='selling_price'))
    y_opt_log = np.log1p(y_opt)

    # Train LightGBM model
    lgb_model = lgb.LGBMRegressor(
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        verbose=-1
    )
    lgb_model.fit(X_opt, y_opt_log)
    print("✅ ML Engine successfully trained & calibrated!")
except Exception as e:
    print(f"☀  Setup failed: {e}")

# --- Step 2: High-Stability Layout Configuration ---
style = {'description_width': '130px'}
layout = widgets.Layout(width='100%', margin='5px 0px')

# Form elements
brand_input = widgets.Combobox(value='Honda', placeholder='e.g., Honda, Maruti', options=sorted(list(wiz_df['brand'].unique())), description='Brand/Make:', style=style, layout=layout)
age_input = widgets.IntSlider(value=8, min=0, max=30, step=1, description='Car Age (Years):', style=style, layout=layout)
km_input = widgets.IntText(value=80000, description='Kilometers Driven:', style=style, layout=layout)
fuel_input = widgets.Dropdown(options=['Petrol', 'Diesel', 'CNG', 'LPG', 'Electric'], value='Petrol', description='Fuel Type:', style=style, layout=layout)
seller_input = widgets.Dropdown(options=['Individual', 'Dealer', 'Trustmark Dealer'], value='Individual', description='Seller Type:', style=style, layout=layout)
trans_input = widgets.Dropdown(options=['Manual', 'Automatic'], value='Automatic', description='Transmission:', style=style, layout=layout)
owner_input = widgets.Dropdown(options=['First Owner', 'Second Owner', 'Third Owner', 'Fourth & Above Owner'], value='First Owner', description='Owner Status:', style=style, layout=layout)
mileage_input = widgets.FloatText(value=12.0, description='Mileage (kmpl):', style=style, layout=layout)
engine_input = widgets.IntText(value=1197, description='Engine (CC):', style=style, layout=layout)
power_input = widgets.IntText(value=117, description='Max Power (bhp):', style=style, layout=layout)
seats_input = widgets.IntSlider(value=5, min=2, max=10, description='Seats:', style=style, layout=layout)
offer_input = widgets.IntText(value=450000, description='Asking Price (Rs):', style=style, layout=layout)

calculate_btn = widgets.Button(description='☁  Calculate Valuation & Deal Range', button_style='success', layout=widgets.Layout(width='100%', height='45px', margin='15px 0px'))
output_panel = widgets.Output(layout=widgets.Layout(width='100%', margin='10px 0px'))

# Construct dual-column input card alignment
left_col = widgets.VBox([brand_input, age_input, km_input, fuel_input, seller_input, trans_input], layout=widgets.Layout(width='48%'))
right_col = widgets.VBox([owner_input, mileage_input, engine_input, power_input, seats_input, offer_input], layout=widgets.Layout(width='48%'))
form_container = widgets.HBox([left_col, right_col], layout=widgets.Layout(width='100%', justify_content='space-between'))

# Dynamic valuation logic tied to button click
def process_valuation(b):
    with output_panel:
        # Clear previous calculation results only (prevents reloading widgets)
        clear_output(wait=True)

        user_car = {
            "brand": brand_input.value.strip(),
            "age": float(age_input.value),
            "km_driven": float(km_input.value),
            "fuel": fuel_input.value,
            "seller_type": seller_input.value,
            "transmission": trans_input.value,
            "owner": owner_input.value,
            "mileage": float(mileage_input.value),
            "engine": float(engine_input.value),
            "max_power": float(power_input.value),
            "seats": float(seats_input.value)
        }

        # Map local features securely
        brand_mapped_price = brand_means.get(user_car["brand"], df_train_orig['selling_price'].mean())
        user_car['brand_tier_price'] = brand_mapped_price
        user_car['power_to_engine'] = user_car['max_power'] / (user_car['engine'] + 1)

        row = pd.get_dummies(pd.DataFrame([user_car]))
        row = row.reindex(columns=X_opt.columns, fill_value=0)

        # Compute prediction
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", UserWarning)
            pred_log = lgb_model.predict(row)
            mid_val = float(np.expm1(pred_log)[0])

        low_val = mid_val * 0.85
        high_val = mid_val * 1.15
        offered_price = float(offer_input.value)
        price_diff_pct = ((offered_price - mid_val) / mid_val) * 100

        # Set verdict styling
        verdict_color = "#34a853"
        if offered_price < low_val:
            verdict = "🔥 VERDICT: EXCELLENT DEAL! The price is well below market average."
            verdict_color = "#ea4335"
        elif offered_price <= mid_val:
            verdict = "✅ VERDICT: GOOD DEAL! The price is very fair and matches our valuation."
            verdict_color = "#34a853"
        elif offered_price <= high_val:
            verdict = "🤝 VERDICT: FAIR PRICE. Standard market rate for the vehicle condition."
            verdict_color = "#fbbc05"
        else:
            verdict = f"❌ VERDICT: OVERPRICED. The price is {price_diff_pct:.1f}% higher than baseline value."
            verdict_color = "#9aa0a6"

        html_report = f"""
        <div style="font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; background: #f8f9fa; border-radius: 8px; border-left: 6px solid {verdict_color}; padding: 20px; box-shadow: 0 2px 10px rgba(0,0,0,0.05);">
            <h3 style="color: #1a73e8; margin-top: 0; margin-bottom: 12px;">🏁 Valuation Report Card</h3>
            <p style="font-size: 14px; margin: 4px 0;"><strong>Car Model:</strong> {user_car['brand'].upper()} ({int(user_car['age'])} Years Old | {int(user_car['km_driven']):,} km driven)</p>
            <p style=